In [1]:
from datasets import load_dataset
from collections import Counter
import gc
import ast
import pandas as pd

In [12]:
full_val_df = pd.read_parquet("data/danbooru2025_val.parquet")
full_val_df

,id,media_asset_created_at,score,rating,media_asset_file_size,tag_string_general,tag_string_character,tag_string_copyright,image_url,tags,image_path
0,3803458,2020-02-29T02:04:09.022-05:00,65,s,623996,1girl antenna_hair armpits bed_sheet blush bre...,al_azif_(demonbane),demonbane,https://cdn.donmai.us/360x360/57/a8/57a82bf4a1...,"[1girl, antenna_hair, armpits, bed_sheet, blus...",data/images/danbooru2025_val/3803458.jpg
1,7615873,2024-05-08T16:52:36.138-04:00,106,e,126497,1boy 1girl all_fours anal ass barefoot black_t...,megumin satou_kazuma,kono_subarashii_sekai_ni_shukufuku_wo!,https://cdn.donmai.us/360x360/04/5e/045e482079...,"[1boy, 1girl, all_fours, anal, ass, barefoot, ...",data/images/danbooru2025_val/7615873.jpg
2,5807689,2022-11-08T06:41:09.289-05:00,99,s,295115,1girl alternate_hairstyle breasts cleavage clo...,arisugawa_natsuha,idolmaster idolmaster_shiny_colors,https://cdn.donmai.us/360x360/dd/58/dd589e8223...,"[1girl, alternate_hairstyle, breasts, cleavage...",data/images/danbooru2025_val/5807689.jpg
3,5771329,2022-10-24T13:40:25.390-04:00,135,e,2100081,1boy 1girl :o arm_grab bar_censor bouncing_bre...,,original,https://cdn.donmai.us/360x360/7e/db/7edb0f7795...,"[1boy, 1girl, :o, arm_grab, bar_censor, bounci...",data/images/danbooru2025_val/5771329.jpg
4,2987219,2018-01-14T16:47:46.466-05:00,62,q,154001,1girl armor black_panties blue_eyes blue_hair ...,nanami_yachiyo,magia_record:_mahou_shoujo_madoka_magica_gaide...,https://cdn.donmai.us/360x360/ff/7e/ff7edf51b7...,"[1girl, armor, black_panties, blue_eyes, blue_...",data/images/danbooru2025_val/2987219.jpg
...,...,...,...,...,...,...,...,...,...,...,...
99995,5599936,2022-08-17T22:49:56.764-04:00,53,s,577595,1girl ahoge animal_ears bare_legs black_skirt ...,,original,https://cdn.donmai.us/360x360/5b/62/5b620ccca4...,"[1girl, ahoge, animal_ears, bare_legs, black_s...",data/images/danbooru2025_val/5599936.jpg
99996,8643081,2024-12-31T10:01:59.979-05:00,143,e,9777813,1boy 1girl :q against_glass anus ass breasts c...,kita_ikuyo,bocchi_the_rock!,https://cdn.donmai.us/360x360/c6/73/c673f9a7f9...,"[1boy, 1girl, :q, against_glass, anus, ass, br...",data/images/danbooru2025_val/8643081.jpg
99997,6042767,2023-02-04T16:56:08.190-05:00,231,e,5237793,1boy 1girl ? animal_ears anus ass bar_censor b...,tsunomaki_watame,hololive,https://cdn.donmai.us/360x360/e9/3a/e93ae304c9...,"[1boy, 1girl, ?, animal_ears, anus, ass, bar_c...",data/images/danbooru2025_val/6042767.jpg
99998,5870341,2022-12-03T09:46:20.261-05:00,52,s,604539,1girl body_freckles bra bracelet breasts cleav...,beelzebub_(helltaker),helltaker,https://cdn.donmai.us/360x360/22/0b/220bd87446...,"[1girl, body_freckles, bra, bracelet, breasts,...",data/images/danbooru2025_val/5870341.jpg


In [13]:
# convert the datetime col
full_val_df["media_asset_created_at"] = pd.to_datetime(
    full_val_df["media_asset_created_at"],
    utc=True,
)

In [14]:
# get all tags from a DF
def get_tags(df_):
    all_tags = Counter()
    for tags in df_["tags"]:
        all_tags.update(tags)
    return all_tags

In [15]:
# all tags before 2022
old_df = full_val_df[full_val_df["media_asset_created_at"].dt.year <= 2021]
old_tags = get_tags(old_df)
len(old_tags)

10445

In [16]:
# all tags after 2023
num_samples = 5000
new_df = full_val_df[full_val_df["media_asset_created_at"].dt.year >= 2023][:num_samples]
new_tags = get_tags(new_df)
len(new_tags)

8439

In [17]:
# get common tags between them
full_tags = {tag for tag in new_tags if tag in old_tags}
len(full_tags)

7612

In [18]:
# most popular tags
stags = sorted((-c, t) for t, c in new_tags.items())
stags = [t.replace("_", " ") for c, t in stags][:1000]
", ".join(stags)


"1girl, breasts, long hair, solo, looking at viewer, blush, large breasts, smile, open mouth, navel, nipples, thighs, simple background, cleavage, black hair, 1boy, hair ornament, blue eyes, white background, bare shoulders, ass, animal ears, shirt, hetero, blue archive, hair between eyes, short hair, gloves, very long hair, underwear, collarbone, thighhighs, nude, multicolored hair, red eyes, sweat, jewelry, censored, halo, long sleeves, penis, blonde hair, swimsuit, closed mouth, sitting, multiple girls, skirt, panties, medium breasts, purple eyes, holding, tail, bikini, pussy, brown hair, standing, heart, dress, blue hair, stomach, sex, original, white shirt, indoors, virtual youtuber, sidelocks, jacket, bow, pink hair, white hair, huge breasts, lying, solo focus, 2girls, grey hair, official alternate costume, alternate costume, pantyhose, twintails, purple hair, green eyes, completely nude, yellow eyes, earrings, choker, cowboy shot, small breasts, ahoge, ponytail, tongue, horns, r

In [19]:
# Keep only common tags
df = new_df.copy()
df["tags"] = df["tags"].apply(
    lambda tags: [tag for tag in tags if tag in full_tags]
)

# Remove images with no remaining tags
df = df[df["tags"].str.len() > 0]

# (Optional) Remove images with only one tag
df = df[df["tags"].str.len() >= 2]

df = df.reset_index(drop=True)
df

,id,media_asset_created_at,score,rating,media_asset_file_size,tag_string_general,tag_string_character,tag_string_copyright,image_url,tags,image_path
0,7615873,2024-05-08 20:52:36.138000+00:00,106,e,126497,1boy 1girl all_fours anal ass barefoot black_t...,megumin satou_kazuma,kono_subarashii_sekai_ni_shukufuku_wo!,https://cdn.donmai.us/360x360/04/5e/045e482079...,"[1boy, 1girl, all_fours, anal, ass, barefoot, ...",data/images/danbooru2025_val/7615873.jpg
1,7711000,2024-06-14 02:28:49.403000+00:00,83,q,3930555,1girl back backboob backless_dress backless_ou...,yor_briar,spy_x_family,https://cdn.donmai.us/360x360/36/15/3615871497...,"[1girl, back, backboob, backless_dress, backle...",data/images/danbooru2025_val/7711000.jpg
2,7570805,2024-05-12 12:43:56.579000+00:00,65,s,264075,1girl ahoge animal_ear_fluff animal_ears back_...,hina_(blue_archive) mash_kyrielight mash_kyrie...,blue_archive fate/grand_order fate_(series),https://cdn.donmai.us/360x360/3b/bb/3bbb87d7ad...,"[1girl, ahoge, animal_ear_fluff, animal_ears, ...",data/images/danbooru2025_val/7570805.jpg
3,7291029,2024-03-04 08:48:56.042000+00:00,54,s,1006806,1girl animal_ears bare_shoulders black_leotard...,hatsune_miku,rabbit_hole_(vocaloid) vocaloid,https://cdn.donmai.us/360x360/c5/48/c548a47499...,"[1girl, animal_ears, bare_shoulders, black_leo...",data/images/danbooru2025_val/7291029.jpg
4,7841062,2024-07-11 14:15:40.595000+00:00,86,q,715030,4girls ace_trainer_(pokemon)_(cosplay) antenna...,ace_trainer_(pokemon) dawn_(pokemon) hilda_(po...,pokemon pokemon_bw pokemon_bw2 pokemon_dppt po...,https://cdn.donmai.us/360x360/22/bb/22bb2ac0e9...,"[4girls, antenna_hair, blush, bow_hairband, br...",data/images/danbooru2025_val/7841062.jpg
...,...,...,...,...,...,...,...,...,...,...,...
4995,6360048,2023-01-23 14:18:05.099000+00:00,93,e,2601736,1girl animal_ears belly blush bottomless bra b...,ookami_mio,hololive,https://cdn.donmai.us/360x360/b2/9f/b29fe368d4...,"[1girl, animal_ears, belly, blush, bottomless,...",data/images/danbooru2025_val/6360048.jpg
4996,6294706,2023-05-09 19:40:58.041000+00:00,382,e,94775,1girl anus ass blush borrowed_character bow bo...,alcremie alcremie_(strawberry_sweet) alcremie_...,pokemon pokemon_swsh,https://cdn.donmai.us/360x360/23/4b/234b995dd7...,"[1girl, anus, ass, blush, borrowed_character, ...",data/images/danbooru2025_val/6294706.jpg
4997,6525406,2023-07-25 05:04:04.555000+00:00,169,e,5202352,1girl anus armpits artist_name ass bag black_b...,satonaka_megumi_(futamotu),original,https://cdn.donmai.us/360x360/43/af/43afcdda95...,"[1girl, anus, armpits, artist_name, ass, bag, ...",data/images/danbooru2025_val/6525406.jpg
4998,5983065,2023-01-14 19:48:32.033000+00:00,68,s,3009462,1girl alternate_costume arm_up armpits blonde_...,durandal_(dea_anchora)_(honkai_impact) duranda...,honkai_(series) honkai_impact_3rd,https://cdn.donmai.us/360x360/e0/1f/e01fce810f...,"[1girl, alternate_costume, arm_up, armpits, bl...",data/images/danbooru2025_val/5983065.jpg


In [20]:
df.to_parquet(
    "data/danbooru_after2023_testset.parquet",
    index=False,
    compression="zstd",
)